# BERT をファインチューニングして日本語の固有表現抽出器を作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_sequence_labeling/bert_ner_finetune.ipynb)

NLP クックブック 15-08 ／ 系列ラベリング

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

日本語の固有表現抽出モデルを、文字位置のアノテーション付きデータから学習したい。

In [ ]:
!pip install -q datasets seqeval accelerate

In [ ]:
import json, urllib.request
from datasets import Dataset

# ストックマーク社の Wikipedia 固有表現データセット (CC BY-SA 3.0)
url = "https://raw.githubusercontent.com/stockmarkteam/ner-wikipedia-dataset/main/ner.json"
data = json.load(urllib.request.urlopen(url))
print(len(data), data[0])

types = sorted({e["type"] for d in data for e in d["entities"]})
labels = ["O"] + [f"{p}-{t}" for t in types for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(labels)}
ds = Dataset.from_list([{"text": d["text"], "starts": [e["span"][0] for e in d["entities"]],
                         "ends": [e["span"][1] for e in d["entities"]], "types": [e["type"] for e in d["entities"]]} for d in data])
ds = ds.train_test_split(test_size=0.2, seed=0)

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification

model_name = "xlm-roberta-base"   # 文字位置 (offset mapping) を返せる高速トークナイザを持つモデルを使う
tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode(example):
    enc = tokenizer(example["text"], truncation=True, max_length=256, return_offsets_mapping=True)
    tags = []
    for s, e in enc["offset_mapping"]:
        if s == e:                                  # 特殊トークンは損失計算から除外
            tags.append(-100)
            continue
        tag = "O"
        for start, end, typ in zip(example["starts"], example["ends"], example["types"]):
            if s < end and e > start:               # トークンがスパンと重なる
                tag = ("B-" if s <= start else "I-") + typ
                break
        tags.append(label2id[tag])
    enc["labels"] = tags
    enc.pop("offset_mapping")
    return enc

tokenized = ds.map(encode, remove_columns=["text", "starts", "ends", "types"])
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label2id)

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score
from transformers import Trainer, TrainingArguments

def to_tags(logits, gold):
    pred = np.argmax(logits, axis=-1)
    y_true = [[labels[g] for g in row if g != -100] for row in gold]
    y_pred = [[labels[p] for p, g in zip(prow, grow) if g != -100] for prow, grow in zip(pred, gold)]
    return y_true, y_pred

def compute_metrics(eval_pred):
    y_true, y_pred = to_tags(*eval_pred)
    return {"f1": f1_score(y_true, y_pred)}

args = TrainingArguments(
    output_dir="ner-out", num_train_epochs=5, learning_rate=5e-5, warmup_steps=60, weight_decay=0.01,
    per_device_train_batch_size=32, per_device_eval_batch_size=64, eval_strategy="epoch", save_strategy="no",
    fp16=True, logging_steps=50, report_to="none", seed=0,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
                  data_collator=DataCollatorForTokenClassification(tokenizer), compute_metrics=compute_metrics)
trainer.train()
out = trainer.predict(tokenized["test"])
print(classification_report(*to_tags(out.predictions, out.label_ids), digits=3))

In [ ]:
from transformers import pipeline

ner = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=0)
text = "ソニーグループの吉田憲一郎会長は、東京都港区の本社で新型プレイステーションを発表した。"
for e in ner(text):
    print(text[e["start"]:e["end"]], e["entity_group"], round(float(e["score"]), 2))

- T4 GPU で 5 エポック約 5 分、エンティティ単位の F1 は 0.85 前後。
- ここではサブワードすべてにラベルを付けている (先頭だけに付けて残りを -100 にする流儀もある)。評価は本来、単語や文字の単位に戻して行うのが厳密。
- 日本語特化の BERT (`tohoku-nlp/bert-base-japanese-v3`) のほうが精度は高いが、トークナイザが文字位置を返さないので、位置合わせを自分で実装する必要がある。